### Jacobian lens — fit and run

In [ ]:
import jlens
import numpy as np 
import os
import pickle

jlens.configure_logging()

MODEL_NAME = 'microsoft/phi-2'
exp_name = 'phi2'

In [ ]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

In [ ]:
### Fitting

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128
)
lens.save("fitted/"+exp_name+"_jlens.pt")

In [ ]:
lens = jlens.JacobianLens.from_pretrained(
    "fitted/"+exp_name+"_jlens.pt"
)
lens

In [ ]:
prompt = "The capital of France is"
jlens_logits, model_logits, _, lens_hidden, final_h = lens.apply(model, prompt, positions=[-1])
logit_lens, _, _, _, _ = lens.apply(model, prompt, positions=[-1], use_jacobian=False)

def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]

for layer in sorted(jlens_logits.keys()):
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
for layer in sorted(jlens_logits.keys()):
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

In [ ]:
jpath = [v.numpy()[0] for k,v in lens_hidden.items()]
jpath.append(final_h.numpy()[0])
jpath = np.array(jpath)
print(jpath.shape)

In [ ]:
dfile = "fitted/"+exp_name+"_jlens_demo1.pkl"
with open(dfile, "wb") as file:
    pickle.dump(jpath, file)